# Example: Operational Dashboard and Stress Report

In this example, we build a daily operational dashboard, run the production simulation across 20 HMM-generated paths to gather aggregate statistics, and inject an extreme crash scenario to test whether the system de-risks and escalates appropriately.

> **By the end of this example, you will be able to:**
> * Build a snapshot dashboard showing current portfolio state, bandit selection, and sentiment
> * Aggregate operational metrics across multiple production simulations
> * Stress-test the production system with a crash injection and evaluate its response

## Setup, Data and Prerequisites

In [ ]:
include("Include.jl");

In [ ]:
let
    # Constants -
    global Δt = 1.0 / 252.0;
    global rf = 0.05;
    global B₀ = 10000.0;
    global N_short = 21;
    global N_long = 63;
    global N_growth = 10;
    global GAIN = 10.0;
    global offset = N_short + N_long;
    global n_production_days = 60;
    global T_total = offset + n_production_days + 20;

    global tickers = ["LargeCap", "SmallCap", "International", "Bond", "Commodity"];
    global sim_params = Dict(
        "LargeCap"      => (0.0002, 1.10, 0.010),
        "SmallCap"      => (0.0003, 1.35, 0.014),
        "International" => (0.0001, 0.95, 0.012),
        "Bond"          => (0.0001, -0.15, 0.003),
        "Commodity"     => (0.0001, 0.60, 0.013)
    );
    global start_prices = Dict("LargeCap" => 150.0, "SmallCap" => 45.0,
        "International" => 80.0, "Bond" => 100.0, "Commodity" => 60.0);

    # Production context -
    global prod_ctx = build(MyProductionContext, (
        tickers = tickers, sim_parameters = sim_params,
        B₀ = B₀, epsilon = 0.1,
        max_drawdown = 0.15, max_turnover = 0.50,
        sentiment_threshold = -0.5, sentiment_override_lambda = 2.0,
        max_bandit_churn = 2
    ));

    # Fit HMM -
    training = generate_training_prices(S₀=100.0, μ=0.08, σ=0.18, T=1260, seed=42);
    global hmm_model = hmm_fit(JumpHiddenMarkovModel, training; rf=rf, N=50, nu=5.0, dt=Δt);
    global hmm_model = hmm_tune(hmm_model, training;
        epsilon_range=range(1e-4, 2.5e-2, length=8),
        lambda_range=range(10.0, 80.0, length=6), n_paths=30);

    println("Setup complete")
end

___
## Task 1: Operational Dashboard
We run a single production simulation and display a daily snapshot dashboard: current holdings, sector weights, drawdown from peak, bandit's current preferred subset, sentiment signal, and days since last rebalance.

> **What should you see?** A comprehensive snapshot of the portfolio's current state, similar to what a portfolio manager would see on their screen each morning.

In [ ]:
let
    # Generate a single path and run production -
    result = hmm_simulate(hmm_model, T_total; n_paths=1);
    G = result.paths[1].observations;
    mkt = JumpHMM.prices_from_growth_rates(G, 100.0; rf=rf, dt=Δt);
    K = length(tickers);

    pmat = zeros(T_total, K + 1);
    pmat[:, 1] = 1:T_total;
    gm_raw = compute_market_growth(mkt; Δt=Δt);
    for (k, ticker) in enumerate(tickers)
        (αᵢ, βᵢ, σᵢ) = sim_params[ticker];
        pmat[1, k + 1] = start_prices[ticker];
        for t in 2:T_total
            gm_t = gm_raw[min(t - 1, length(gm_raw))];
            gᵢ = αᵢ + βᵢ * gm_t * Δt + σᵢ * sqrt(Δt) * randn();
            pmat[t, k + 1] = pmat[t - 1, k + 1] * exp(gᵢ);
        end
    end

    ema_s = compute_ema(mkt; window=N_short);
    ema_l = compute_ema(mkt; window=N_long);
    λ = compute_lambda(ema_s, ema_l; G=GAIN);
    λ[1:offset] .= 0.0;
    gm_e = compute_ema(gm_raw; window=N_growth);
    sent = generate_synthetic_sentiment(mkt; noise_σ=0.15, seed=88);

    (results, events) = run_production_simulation(
        prod_ctx, mkt, pmat, sent, λ, gm_e;
        n_days=n_production_days, offset=offset, n_bandit_iters=80);

    global dash_results = results;
    global dash_events = events;

    # Display dashboard for the last day -
    last = results[end];
    actual_day = offset + last.day;
    prices_now = [pmat[actual_day, k + 1] for k in 1:K];
    values = [last.shares[k] * prices_now[k] for k in 1:K];
    total_invested = sum(values);
    weights = total_invested > 0 ? values ./ total_invested : zeros(K);

    println("═"^60)
    println("  OPERATIONAL DASHBOARD — Day $(last.day)")
    println("═"^60)
    println("  Total Wealth:     \$$(round(last.wealth, digits=2))")
    println("  Cash:             \$$(round(last.cash, digits=2))")
    println("  Invested:         \$$(round(total_invested, digits=2))")
    println("  Sentiment:        $(round(last.sentiment, digits=3))")
    println("  Effective λ:      $(round(last.lambda, digits=3))")
    println("  Rebalanced today: $(last.rebalanced)")
    println("  Escalated today:  $(last.escalated)")
    println()

    # Holdings table -
    selected = last.bandit_action .== 1;
    holdings_df = DataFrame(
        "Ticker" => tickers,
        "Shares" => round.(last.shares, digits=2),
        "Price" => round.(prices_now, digits=2),
        "Value" => round.(values, digits=2),
        "Weight (%)" => round.(weights .* 100, digits=1),
        "Selected" => [s ? "Yes" : "No" for s in selected]
    );
    pretty_table(holdings_df, tf=tf_markdown)
end

**Visualize:** Dashboard overview — wealth, sentiment, and bandit selection.

In [ ]:
let
    days = [r.day for r in dash_results];
    wealth = [r.wealth for r in dash_results];
    sentiments = [r.sentiment for r in dash_results];
    K = length(tickers);

    # Wealth with drawdown fill -
    peak = accumulate(max, wealth);
    p1 = plot(days, wealth, label="Wealth", linewidth=2, color=:steelblue, fill=(peak, 0.1, :red))
    plot!(p1, days, peak, label="Peak", linewidth=1, color=:gray50, linestyle=:dot)
    ylabel!(p1, "Wealth (\$)")
    title!(p1, "Portfolio Wealth (shaded = drawdown)")

    # Sentiment -
    p2 = plot(days, sentiments, label="Sentiment", linewidth=2, color=:purple)
    hline!(p2, [prod_ctx.sentiment_threshold], label="Threshold", color=:red, linestyle=:dash)
    ylabel!(p2, "Score")
    title!(p2, "Sentiment")

    # Bandit selection heatmap -
    n = length(days);
    inclusion = zeros(n, K);
    for (i, r) in enumerate(dash_results)
        inclusion[i, :] = r.bandit_action;
    end
    p3 = heatmap(days, 1:K, inclusion', color=[:white, :steelblue], clim=(0, 1),
        yticks=(1:K, tickers), xlabel="Production Day", colorbar=false)
    title!(p3, "Bandit Selection (blue = included)")

    plot(p1, p2, p3, layout=(3, 1), size=(800, 700), legend=:topright)
end

___
## Task 2: Multi-Path Production Simulation
We run the production system across 20 HMM-generated paths to gather aggregate operational metrics: how often does the bandit change its subset? How often do escalation triggers fire? What's the distribution of final wealth?

> **What should you see?** Aggregate statistics that reveal the system's typical operational behavior across diverse market conditions. Some paths will trigger escalations; others will run smoothly.

In [ ]:
let
    n_paths = 20;
    K = length(tickers);

    # Storage for aggregate metrics -
    all_metrics = Dict{String,Any}[];
    all_final_wealth = Float64[];
    all_max_dd = Float64[];
    all_n_escalations = Int[];
    all_bandit_changes = Int[];

    println("Running production simulation across $(n_paths) paths...")

    for p in 1:n_paths

        # Generate path -
        result = hmm_simulate(hmm_model, T_total; n_paths=1);
        G = result.paths[1].observations;
        mkt = JumpHMM.prices_from_growth_rates(G, 100.0; rf=rf, dt=Δt);

        pmat = zeros(T_total, K + 1);
        pmat[:, 1] = 1:T_total;
        gm_raw = compute_market_growth(mkt; Δt=Δt);
        for (k, ticker) in enumerate(tickers)
            (αᵢ, βᵢ, σᵢ) = sim_params[ticker];
            pmat[1, k + 1] = start_prices[ticker];
            for t in 2:T_total
                gm_t = gm_raw[min(t - 1, length(gm_raw))];
                gᵢ = αᵢ + βᵢ * gm_t * Δt + σᵢ * sqrt(Δt) * randn();
                pmat[t, k + 1] = pmat[t - 1, k + 1] * exp(gᵢ);
            end
        end

        ema_s = compute_ema(mkt; window=N_short);
        ema_l = compute_ema(mkt; window=N_long);
        λ = compute_lambda(ema_s, ema_l; G=GAIN);
        λ[1:offset] .= 0.0;
        gm_e = compute_ema(gm_raw; window=N_growth);
        sent = generate_synthetic_sentiment(mkt; noise_σ=0.15);

        (results, events) = run_production_simulation(
            prod_ctx, mkt, pmat, sent, λ, gm_e;
            n_days=n_production_days, offset=offset, n_bandit_iters=80);

        metrics = compute_dashboard_metrics(results, events);
        push!(all_metrics, metrics);
        push!(all_final_wealth, metrics["final_wealth"]);
        push!(all_max_dd, metrics["max_drawdown"]);
        push!(all_n_escalations, metrics["n_escalations"]);
        push!(all_bandit_changes, metrics["n_bandit_changes"]);
    end

    global _all_final_wealth = all_final_wealth;
    global _all_max_dd = all_max_dd;
    global _all_metrics = all_metrics;

    println("\n" * "═"^60)
    println("  AGGREGATE OPERATIONAL METRICS ($(n_paths) paths)")
    println("═"^60)

    agg_df = DataFrame(
        "Metric" => ["Median Final Wealth", "Median Max Drawdown (%)",
            "Avg Escalations per Path", "Avg Bandit Changes per Path",
            "Paths with Escalations (%)"],
        "Value" => [
            round(median(all_final_wealth), digits=0),
            round(median(all_max_dd) * 100, digits=1),
            round(mean(all_n_escalations), digits=1),
            round(mean(all_bandit_changes), digits=1),
            round(sum(all_n_escalations .> 0) / n_paths * 100, digits=0)
        ]
    );
    pretty_table(agg_df, tf=tf_markdown)
end

**Visualize:** Distribution of final wealth and escalation counts across paths.

In [ ]:
let
    p1 = histogram(_all_final_wealth, bins=10, label="Final Wealth", color=:steelblue, alpha=0.7)
    vline!(p1, [B₀], label="Starting Capital", linestyle=:dash, color=:black, linewidth=2)
    xlabel!(p1, "Final Wealth (\$)")
    ylabel!(p1, "Count")
    title!(p1, "Production Final Wealth Distribution")

    p2 = histogram(_all_max_dd .* 100, bins=10, label="Max Drawdown", color=:coral, alpha=0.7)
    vline!(p2, [prod_ctx.max_drawdown * 100], label="DD Limit", linestyle=:dash, color=:red, linewidth=2)
    xlabel!(p2, "Max Drawdown (%)")
    ylabel!(p2, "Count")
    title!(p2, "Production Max Drawdown Distribution")

    plot(p1, p2, layout=(1, 2), size=(900, 400))
end

___
## Task 3: Crash Stress Test
We inject an extreme regime shift into a production path — a sudden 30% market crash starting at day 30 — and observe whether the system responds appropriately. Does it de-risk? Does the escalation trigger fire? Does the bandit adapt?

> **What should you see?** The crash should cause the drawdown trigger to fire (critical escalation), moving the portfolio to cash. The sentiment signal should plummet. The bandit should adapt its asset selection post-crash. This is the ultimate test of the production safety net.

In [ ]:
let
    K = length(tickers);

    # Generate a normal path -
    result = hmm_simulate(hmm_model, T_total; n_paths=1);
    G = result.paths[1].observations;
    mkt = JumpHMM.prices_from_growth_rates(G, 100.0; rf=rf, dt=Δt);

    # Inject crash: 30% drop over 5 days starting at offset + 30 -
    crash_start = offset + 30;
    crash_days = 5;
    crash_factor = 0.70;  # 30% drop
    daily_drop = crash_factor^(1.0 / crash_days);

    mkt_stressed = copy(mkt);
    for d in crash_start:(crash_start + crash_days - 1)
        if d <= T_total
            mkt_stressed[d] = mkt_stressed[d-1] * daily_drop;
        end
    end
    # keep post-crash level -
    for d in (crash_start + crash_days):T_total
        ratio = mkt_stressed[crash_start + crash_days - 1] / mkt[crash_start + crash_days - 1];
        mkt_stressed[d] = mkt[d] * ratio;
    end

    # Build per-ticker prices from stressed market -
    pmat = zeros(T_total, K + 1);
    pmat[:, 1] = 1:T_total;
    gm_raw = compute_market_growth(mkt_stressed; Δt=Δt);
    for (k, ticker) in enumerate(tickers)
        (αᵢ, βᵢ, σᵢ) = sim_params[ticker];
        pmat[1, k + 1] = start_prices[ticker];
        for t in 2:T_total
            gm_t = gm_raw[min(t - 1, length(gm_raw))];
            gᵢ = αᵢ + βᵢ * gm_t * Δt + σᵢ * sqrt(Δt) * randn();
            pmat[t, k + 1] = pmat[t - 1, k + 1] * exp(gᵢ);
        end
    end

    ema_s = compute_ema(mkt_stressed; window=N_short);
    ema_l = compute_ema(mkt_stressed; window=N_long);
    λ = compute_lambda(ema_s, ema_l; G=GAIN);
    λ[1:offset] .= 0.0;
    gm_e = compute_ema(gm_raw; window=N_growth);
    sent = generate_synthetic_sentiment(mkt_stressed; noise_σ=0.10, seed=55);

    # Run production -
    (stress_results, stress_events) = run_production_simulation(
        prod_ctx, mkt_stressed, pmat, sent, λ, gm_e;
        n_days=n_production_days, offset=offset, n_bandit_iters=80);

    global _stress_results = stress_results;
    global _stress_events = stress_events;

    # Report -
    metrics = compute_dashboard_metrics(stress_results, stress_events);

    println("═"^60)
    println("  STRESS TEST: 30% Crash at Day 30")
    println("═"^60)
    println("  Final wealth:         \$$(round(metrics["final_wealth"], digits=2))")
    println("  Max drawdown:         $(round(metrics["max_drawdown"]*100, digits=1))%")
    println("  Escalation events:    $(metrics["n_escalations"]) ($(metrics["n_critical"]) critical)")
    println("  Bandit subset changes: $(metrics["n_bandit_changes"])")

    # Pass/fail assessment -
    println("\n  Production Readiness Assessment:")
    dd_ok = metrics["max_drawdown"] <= 0.25;
    esc_ok = metrics["n_critical"] > 0;  # system SHOULD escalate during a crash
    println("    Drawdown contained (≤25%):    $(dd_ok ? "PASS" : "FAIL") ($(round(metrics["max_drawdown"]*100, digits=1))%)")
    println("    System escalated during crash: $(esc_ok ? "PASS" : "FAIL") ($(metrics["n_critical"]) critical events)")
    println("    Capital preserved (>\$8000):   $(metrics["final_wealth"] > 8000 ? "PASS" : "FAIL") (\$$(round(metrics["final_wealth"], digits=0)))")
end

**Visualize:** Stress test — wealth curve with crash zone and escalation markers.

In [ ]:
let
    days = [r.day for r in _stress_results];
    wealth = [r.wealth for r in _stress_results];
    sentiments = [r.sentiment for r in _stress_results];

    # Wealth with crash zone -
    p1 = plot(days, wealth, label="Wealth", linewidth=2.5, color=:steelblue)
    vspan!(p1, [30, 35], alpha=0.2, color=:red, label="Crash Zone")
    hline!(p1, [B₀], linestyle=:dash, color=:gray50, label="Starting Capital")
    for evt in _stress_events
        if evt.severity == :critical
            vline!(p1, [evt.day], color=:red, alpha=0.5, linewidth=2, label="")
        end
    end
    ylabel!(p1, "Wealth (\$)")
    title!(p1, "Stress Test: 30% Crash at Day 30")

    # Sentiment during crash -
    p2 = plot(days, sentiments, label="Sentiment", linewidth=2, color=:purple)
    vspan!(p2, [30, 35], alpha=0.2, color=:red, label="")
    hline!(p2, [prod_ctx.sentiment_threshold], linestyle=:dash, color=:red, label="Threshold")
    xlabel!(p2, "Production Day")
    ylabel!(p2, "Score")
    title!(p2, "Sentiment During Crash")

    plot(p1, p2, layout=(2, 1), size=(800, 550), legend=:topright)
end

___
## Summary
In this example, we completed the production operations toolkit:

- The **operational dashboard** provides a daily snapshot of portfolio state, bandit selection, and sentiment
- **Multi-path simulation** reveals typical operational metrics: escalation frequency, bandit churn, and wealth distribution
- The **crash stress test** validates that the production safety net works: the system detects the crash, fires escalation triggers, and de-risks to protect capital

This completes the four-session AI-in-Finance certificate:
1. **Session 1:** Classical portfolio construction and its failure modes
2. **Session 2:** Cobb-Douglas utility maximization with sentiment-driven allocation
3. **Session 3:** HMM regime-switching backtesting with bandit challengers
4. **Session 4:** Production deployment with operational safeguards

### Disclaimer
This content is for educational purposes only and does not constitute investment advice.